# PhysioPain v2 Dataset Analysis & Prediction
**Dataset**: `synthetic_physiopain_1to8_v2.csv`

This notebook implements the complete ML pipeline:
1. **EDA**: Visualizing distributions and correlations.
2. **Feature Engineering**: Vector magnitudes and interaction features.
3. **Model Selection**: Random Forest Classifier for pain level prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv('new_data.csv')
print(f"Dataset Loaded: {df.shape[0]} samples, {df.shape[1]} features.")
display(df.head())

## 1. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Pain Level Distribution
target_col = 'pain_level'
sns.countplot(data=df, x=target_col, palette='viridis', ax=ax[0])
ax[0].set_title('Pain Level Distribution (1-8)')

# Correlation Matrix
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='RdYlGn', ax=ax[1])
ax[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots of key features by Pain Level
features_to_plot = ['eda', 'hr', 'temp']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(features_to_plot):
    sns.boxplot(data=df, x='pain_level', y=feat, ax=axes[i], palette='magma')
    axes[i].set_title(f'{feat.upper()} vs Pain Level')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
def engineer_features(data):
    df_fe = data.copy()
    
    # 1. Acceleration Magnitude (Movement State)
    df_fe['acc_magnitude'] = np.sqrt(df_fe['acc_x']**2 + df_fe['acc_y']**2 + df_fe['acc_z']**2)
    
    # 2. Stress Index interaction (EDA * HR)
    df_fe['stress_index'] = df_fe['eda'] * df_fe['hr']
    
    # 3. Temp Delta (Variation from mean Body Temp)
    df_fe['temp_delta'] = np.abs(df_fe['temp'] - df_fe['temp'].mean())
    
    return df_fe

print("Performing advanced feature engineering...")
df_proc = engineer_features(df)
print(f"Feature space expanded: {df_proc.shape[1]} columns.")

## 3. Machine Learning: Random Forest

In [ ]:
X = df_proc.drop(columns=['pain_level'])
y = df_proc['pain_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training optimized Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

y_preds = rf.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_preds)

print(f"\nSystem Validation Accuracy: {accuracy*100:.2f}%")

In [ ]:
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Prediction Confusion Matrix')
plt.ylabel('Actual Pain Level')
plt.xlabel('Predicted Pain Level')
plt.show()

print("Full Metrics Report:")
print(classification_report(y_test, y_preds))

In [ ]:
feat_importances = pd.Series(rf.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
feat_importances.nlargest(10).plot(kind='barh', color='teal')
plt.title('Top 10 Important Features for Pain Prediction')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.show()